In [ ]:
import json
from pathlib import Path

import pandas as pd
from claude_agent_sdk import ClaudeAgentOptions, query
from claude_agent_sdk.types import ResultMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

In [ ]:
train = pd.read_parquet("../data/processed/corpora.parquet")
test = pd.read_parquet("../data/processed/evaluation.parquet")

#### Commodity Description Generation

In [ ]:
class Entity(BaseModel):
    subheading: str = Field(
        description="the 6-digit HS subheading code, copied exactly",
    )
    names: list[str] = Field(
        description=f"5 short product names for goods under it",
    )


parser = PydanticOutputParser(pydantic_object=Entity)

prompt_sys = (
    "You write the kind of short product names an online shopper or a customs "
    "declarant 1 to 4 words, everyday vocabulary, brand-free."
)

prompt_usr = PromptTemplate.from_template(
    "For each HS subheading below, list {n} distinct NEW names for goods that "
    "fall under it. Do not repeat the phrases you are given."
    "\n\n{data}"
    "\n\n{format_instructions}",
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
        "n": 5,
    },
)

In [ ]:
async def claude(prompt_value):
    opts = ClaudeAgentOptions(
        system_prompt=prompt_sys,
        model="opus",
        max_turns=1,
        tools=[],
    )
    text = ""
    async for m in query(prompt=prompt_value.to_string(), options=opts):
        if isinstance(m, ResultMessage):
            text = m.result or ""
    return text

chain = prompt_usr | RunnableLambda(claude) | parser
chain = chain.with_retry(stop_after_attempt=3)

In [ ]:
def fetch_descriptions(code):
    df = train.query("subheading == @code")
    legal = "; ".join(df.query("field == 'description'")["desc"].head(3).values)
    known = "; ".join(df.query("field == 'commodity'")["desc"].head(8).values)
    return f"{code}\n  legal: {legal}\n  known: {known}"


def append_synthetic_queries(result):
    with Path("../data/interim/synthetic.jsonl").open("a") as file:
        for name in result.names:
            file.write(
                json.dumps({
                    "subheading": result.subheading,
                    "desc": name.strip().lower(),
                }) + "\n"
            )

In [ ]:
for code in train["subheading"].unique():
    data = fetch_descriptions(code)
    result = await chain.ainvoke({"data": data})
    append_synthetic_queries(result)

#### Data Cleansing

In [ ]:
synthetic = pd.read_json(
    "../data/interim/synthetic.jsonl",
    lines=True,
    dtype=str,
)

synthetic.query("desc not in @train.desc", inplace=True)
synthetic.query("desc not in @test.desc", inplace=True)
synthetic.drop_duplicates(inplace=True, ignore_index=True)

synthetic[["chapter", "heading"]] = pd.concat(
    [synthetic["subheading"].str[:n] for n in (2, 4)],
    axis=1,
).to_numpy()
synthetic["hkhs"] = synthetic["subheading"] + "00"
synthetic["field"] = "synthetic"

synthetic = synthetic[[
    "chapter", "heading", "subheading", "hkhs",
    "desc", "field",
]]

In [ ]:
synthetic.to_parquet("../data/processed/synthetic.parquet", index=False)